# 01 — OlympiadBench-Math-Ko baseline evaluation
Evaluate the unmodified EXAONE base model on all 674 Korean test problems from `ChuGyouk/OlympiadBench-Math-Ko`.

In [ ]:
REPO_URL = "https://github.com/seungjun-green/Korean-TDCS"
!git clone {REPO_URL} korean-math-tdcs
%cd korean-math-tdcs
!pip install -e .

In [ ]:
import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

drive_results = Path("/content/drive/MyDrive/Korean-TDCS/results")
local_results = Path.cwd() / "results"
drive_results.mkdir(parents=True, exist_ok=True)

if local_results.is_symlink():
    if local_results.resolve() != drive_results.resolve():
        raise RuntimeError(f"{local_results} points to the wrong Drive directory")
elif local_results.exists():
    shutil.copytree(local_results, drive_results, dirs_exist_ok=True)
    shutil.rmtree(local_results)

if not local_results.exists():
    local_results.symlink_to(drive_results, target_is_directory=True)

print(f"Saving all outputs to {drive_results}")

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

In [ ]:
EVAL_BATCH_SIZE = 16  # Increase only if it fits in GPU memory.
MAX_NEW_TOKENS = 4096  # Increase if the token-limit rate remains high.
BASELINE_RESULTS_PATH = (
    "results/baseline/olympiad_bench_math_ko/"
    f"max_tokens_{MAX_NEW_TOKENS}/metrics.json"
)

cmd = ("python scripts/evaluate.py --config configs/baseline.yaml "
       f"--set evaluation.batch_size={EVAL_BATCH_SIZE} "
       f"--set evaluation.generation.max_new_tokens={MAX_NEW_TOKENS} "
       f"--set output.results_path={BASELINE_RESULTS_PATH}")
!{cmd}

In [ ]:
import json

import pandas as pd

result = json.load(open(BASELINE_RESULTS_PATH))
pd.DataFrame({k: v for k, v in result['benchmarks'].items()}).T